# 2. Tiền xử lý dữ liệu

**Input:** `exps/data/data_EDA.csv`  
**Output:** `exps/feature1/` gồm `x_train.csv, y_train.csv, x_test.csv, test_id.csv, info.json`

BEGIN

In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import sys; sys.path.insert(0, "../../extra")
from config import *
from common import *
from utils import *
display.clear_output()

In [3]:
MEMBER   = "TV1"      # tên/mã thành viên -> file log riêng
NOTEBOOK = "pre-processing1"
FEATURE_SET = "feature1"   # thư mục output trong exps/
seed_everything()

## VIỆC CỦA TV1 — tạo `feature1`
1. **NaN = “không có”:** điền `"None"` cho BsmtQual, BsmtCond, BsmtExposure, BsmtFinType1/2, FireplaceQu, Garage*, MasVnrType...; điền 0 cho MasVnrArea, BsmtFinSF1/2, TotalBsmtSF, GarageCars, GarageArea... Khi đó **giữ lại** Alley/PoolQC/Fence/MiscFeature thay vì xóa.
2. **Ngoại lai:** bỏ các dòng train có GrLivArea > 4000 (mục 2.2).
3. **Mã hóa thứ bậc:** Ex/Gd/TA/Fa/Po → 5..1 (None = 0) cho các cột chất lượng, bỏ chúng khỏi `cat_cols`.
4. Mỗi thay đổi: chạy lại notebook này rồi chạy nhanh `4.model/traditional model.ipynb` với `FEATURE_SET="feature1"` để đo tác động; ghi vào bảng cho báo cáo Chương 3.

In [4]:
df, numeric_columns, category_columns = load_eda()
df.shape

(2919, 82)

## 2.1. Làm sạch
Bỏ 4 cột thiếu nhiều, cột phân loại điền mode. Cột số để NaN — `make_pipeline()` ở stage 4 sẽ điền median **trong từng fold**.

In [5]:
# ===== CÔNG TẮC: bật lần lượt từng bước, mỗi lần Run All rồi ghi kết quả =====
NAN_SEMANTIC = True    # Bước 1: NaN = "không có"
DROP_OUTLIER = True    # Bước 2: loại GrLivArea > 4000
ORDINAL      = True    # Bước 3: mã hóa thứ bậc cột chất lượng
STEP_NAME = 'b1+b2+b3' # tên hiển thị trong log, sửa theo công tắc đang bật

# Cột phân loại: NaN nghĩa là "không có" (theo data_description.txt)
NONE_COLS = ['Alley', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
             'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
             'PoolQC', 'Fence', 'MiscFeature', 'MasVnrType']
# Cột số: không có hầm/gara/ốp tường -> diện tích, số lượng = 0
ZERO_COLS = ['MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
             'BsmtFullBath', 'BsmtHalfBath', 'GarageCars', 'GarageArea']

QUAL = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
ORD_MAP = {c: QUAL for c in ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC',
                             'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC']}
ORD_MAP.update({
    'BsmtExposure': {'None': 0, 'No': 1, 'Mn': 2, 'Av': 3, 'Gd': 4},
    'BsmtFinType1': {'None': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6},
    'BsmtFinType2': {'None': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6},
    'GarageFinish': {'None': 0, 'Unf': 1, 'RFn': 2, 'Fin': 3},
})

if NAN_SEMANTIC:
    for c in NONE_COLS:
        df[c] = df[c].fillna('None')
    for c in ZERO_COLS:
        df[c] = df[c].fillna(0)
    df['GarageYrBlt'] = df['GarageYrBlt'].fillna(df['YearBuilt'])
    DROP_COLS = []                                   # giữ lại 4 cột thiếu nhiều
else:
    DROP_COLS = ['Alley', 'PoolQC', 'Fence', 'MiscFeature']
df = df.drop(columns=DROP_COLS)

df['MSSubClass'] = df['MSSubClass'].astype(str)       # mã loại nhà, không phải số
cat_cols = [c for c in category_columns if c not in DROP_COLS] + ['MSSubClass']

if ORDINAL:
    for c, mapping in ORD_MAP.items():
        if c in df.columns:
            df[c] = df[c].fillna('None').map(mapping).fillna(0).astype(int)
    cat_cols = [c for c in cat_cols if c not in ORD_MAP]

for c in cat_cols:                                    # còn NaN thật -> mode của train
    df[c] = df[c].fillna(df.loc[df.is_train == 1, c].mode()[0])
print(len(cat_cols), 'cột one-hot | NaN còn lại trong cột phân loại:', df[cat_cols].isna().sum().sum())

30 cột one-hot | NaN còn lại trong cột phân loại: 0


## 2.2. Ngoại lai

In [6]:
if DROP_OUTLIER:
    outlier_mask = (df.is_train == 1) & (df.GrLivArea > 4000)
else:
    outlier_mask = pd.Series(False, index=df.index)
print('Loại', outlier_mask.sum(), 'dòng')
df = df[~outlier_mask].reset_index(drop=True)
print('Số dòng train:', df.is_train.sum())

Loại 4 dòng
Số dòng train: 1456


## 2.4. Mã hóa & target
One-hot trên train+test gộp (như code tham khảo) để hai tập có cùng cột.

In [7]:
y_train  = np.log1p(df.loc[df.is_train == 1, 'SalePrice'].values)
test_id  = df.loc[df.is_train == 0, 'Id'].values
X = pd.get_dummies(df.drop(columns=['Id', 'SalePrice']), columns=cat_cols, dtype=int)
x_train = X[X.is_train == 1].drop(columns='is_train').reset_index(drop=True)
x_test  = X[X.is_train == 0].drop(columns='is_train').reset_index(drop=True)
x_train.shape, x_test.shape

((1456, 257), (1459, 257))

In [8]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor

assert len(x_train) == len(y_train)
for m_name, m in {'Ridge': Ridge(alpha=10),
                  'GBR': GradientBoostingRegressor(n_estimators=500, learning_rate=0.05,
                                                   max_depth=3, random_state=SEED)}.items():
    scores, _ = evaluate(m, x_train, y_train)
    print(m_name, 'RMSE từng fold:', scores.round(4))
    log_experiment(MEMBER, NOTEBOOK, FEATURE_SET, m_name, scores, selection=STEP_NAME, note='đo bước tiền xử lý')

Ridge RMSE từng fold: [0.126  0.1308 0.1214 0.1327 0.1207]
[log] Ridge        | feature1/b1+b2+b3 | CV RMSE = 0.12636 ± 0.00483
GBR RMSE từng fold: [0.1274 0.124  0.1318 0.123  0.104 ]
[log] GBR          | feature1/b1+b2+b3 | CV RMSE = 0.12205 ± 0.00953


## 2.5. Lưu

In [9]:
save_feature_set(FEATURE_SET, x_train, y_train, x_test, test_id,
                 info={'notebook': NOTEBOOK, 'member': MEMBER, 'note': 'mô tả ngắn thay đổi so với feature0'})

[save] d:/HocSau/House_Prices/house_prices_project/exps/feature1: x_train (1456, 257), x_test (1459, 257)


### Kết luận
- Loại 4 căn GrLivArea > 4000 là bước hiệu quả nhất (Ridge 0.150 → 0.129, std giữa các fold 0.045 → 0.007). Điền NaN theo ngữ nghĩa một mình chưa cải thiện vì ngoại lai còn chi phối; mã hóa thứ bậc giảm số cột 316 → 257 và giúp thêm mô hình tuyến tính.